# 02 - Train the four methods and compare them properly

All four ideologies run through **one** training loop
(`evissl/engine/trainer.py`), so `semi.method` is the only thing that differs.
Anything else - separate scripts per method - would leave a difference in
results attributable to an incidental difference in the schedule rather than to
the ideology under test.

The comparison ends in paired statistics, not a table of means: with 150 test
images and per-image Dice standard deviations near 0.1, a gap of a few points is
inside the noise unless it is tested.

**Runtime.** The settings below are CPU-scale and take roughly two hours for all
four runs on two cores. Set `QUICK = True` for a few minutes at reduced fidelity,
or run `05_colab_full_run.ipynb` on a GPU for full scale.

In [ ]:
# Run from the repository root, or from notebooks/ - both work.
import sys, os
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
    os.chdir(root)
sys.path.insert(0, str(root / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

torch.set_num_threads(2)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
print("repo root :", root)
print("torch     :", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
QUICK = False   # True -> minutes instead of hours, at lower fidelity

CPU_SCALE = [
    "data.image_size=64", "data.train_size=600", "data.val_size=80",
    "data.test_size=150", "data.labeled_fraction=0.10",
    "data.batch_size=8", "data.mu=2",
    "model.width=16", "model.depth=3",
    "optim.epochs=24", "optim.steps_per_epoch=24",
    "loss.kl_anneal_epochs=10", "semi.rampup_epochs=8",
    "run.out_dir=results/scratch",
]
QUICK_SCALE = [
    "data.image_size=48", "data.train_size=160", "data.val_size=32",
    "data.test_size=48", "data.labeled_fraction=0.15",
    "data.batch_size=6", "data.mu=1",
    "model.width=12", "model.depth=3",
    "optim.epochs=6", "optim.steps_per_epoch=8",
    "loss.kl_anneal_epochs=3", "semi.rampup_epochs=2", "eval.bootstrap=500",
    "run.out_dir=results/scratch",
]
OVERRIDES = QUICK_SCALE if QUICK else CPU_SCALE

METHODS = [
    "configs/supervised_baseline.yaml",   # lower bound: labelled data only
    "configs/mean_teacher.yaml",          # prior work: unweighted consistency
    "configs/fixmatch.yaml",              # prior work: hard confidence threshold
    "configs/evidential.yaml",            # ours: vacuity gates, dissonance tempers
]
print("\n".join(OVERRIDES))

## Before training: what does each model cost?

Measured, not quoted. Note that the MAC reduction and the latency reduction are
*not* the same number - dense convolutions vectorise well on CPU while the
narrow depthwise convolutions that produce the MAC saving are memory-bound.

In [ ]:
from evissl.pipelines import run_efficiency_benchmark

efficiency = run_efficiency_benchmark(
    ("separable_unet_tiny", "separable_unet", "unet"),
    out_dir="results/scratch",
    image_size=64 if QUICK else 128,
    batch_sizes=(1,),
    repeats=15,
)
display(efficiency[[
    "model", "params_m", "gmacs", "latency_bs1_ms",
    "params_reduction_x", "macs_reduction_x", "latency_reduction_x", "macs_per_ms_M",
]].round(3))

## Train

Each run writes `config.yaml`, `history.jsonl`, `per_image.csv` and
`summary.json` under `results/runs/<name>/`, plus a checkpoint carrying both the
student and the EMA teacher.

In [ ]:
from evissl.pipelines import run_comparison

comparison = run_comparison(
    METHODS,
    out_dir="results/scratch",
    overrides=OVERRIDES,
    baseline="supervised_baseline",
    metrics=("dice", "iou", "hd95", "boundary_f1"),
    keep_predictions=True,   # needed for the calibration and qualitative figures
)
print("done")

## The headline table

In [ ]:
from evissl.pipelines import format_table

table = comparison["table"]
display(table[[c for c in ("run", "dice", "iou", "hd95", "assd", "boundary_f1",
                           "ece", "ause", "params_m", "latency_ms")
               if c in table.columns]].round(4))
print()
print(format_table(table))

## Is the difference real?

Wilcoxon signed-rank on per-image values, a paired bootstrap interval on the
mean difference, and Holm-Bonferroni correction across the family. An interval
excluding zero is the claim; a mean is not.

In [ ]:
for metric, comparisons in comparison["comparisons"].items():
    print(f"--- {metric} (vs supervised_baseline, Holm-corrected) ---")
    for c in comparisons:
        print("   ", c.summary())
    print()

In [ ]:
from evissl.report import write_comparison_figures, write_markdown_report

written = write_comparison_figures(comparison, "results/scratch")
report = write_markdown_report(comparison, "results/scratch/RESULTS_generated.md")
print(f"{len(written)} figures written")
for path in written:
    print("  ", path)
print("report:", report)

## Training dynamics

In [ ]:
from evissl.viz import plot_training_curves

fig = plot_training_curves(
    comparison["histories"],
    metrics=("train_loss", "val_dice", "train_mask_rate", "train_mean_vacuity"),
)
plt.show()

`train_mask_rate` is the interesting curve. It is the effective fraction of
unlabelled pixels each rule actually used, on a comparable scale across methods:
literally the retained fraction for FixMatch, and the mean belief-mass weight
for the evidential rule. If FixMatch sits near zero for the early epochs, it
spent them learning from labelled data alone.

In [ ]:
from evissl.viz import plot_method_comparison

for metric, intervals in comparison["intervals"].items():
    fig = plot_method_comparison(intervals, metric,
                                 lower_is_better=metric in {"hd95", "assd"})
    plt.show()

Next: **03_ablations.ipynb** removes one mechanism at a time, and
**04_calibration_and_uncertainty.ipynb** asks whether the uncertainty is any
good rather than just whether the segmentation is.